# About: CoursewareHubインベントリの準備

CoursewareHubに必要なソフトウェアの設定は [Ansible](https://www.ansible.com/) を利用します。

これから設定するCoursewareHubに関する各種設定を、インベントリ ... 特に [Ansibleグループ変数](https://docs.ansible.com/ansible/latest/user_guide/playbooks_variables.html) として定義します。

以下のようなフォルダ、ファイルにAnsible関連ファイルを配置します。

- Inventoryファイルは `./hosts`
- グループ変数定義ディレクトリは `./group_vars` とします。

# 準備

ファイルの準備をします。

In [1]:
import os

group_vars_dir = './group_vars'
if not os.path.exists(group_vars_dir):
    os.mkdir(group_vars_dir)
!ls -la {group_vars_dir}

total 8
drwxr-xr-x. 2 niiyazawa users   25 Sep 10 09:16 .
drwxr-xr-x. 8 niiyazawa  1002 4096 Sep 13 08:55 ..
-rw-r--r--. 1 niiyazawa users  342 Sep 10 09:16 cwhtest0001


# 設定対象サーバーの指定

設定対象のサーバーは、すでに `01_AWS VMの作成.ipynb` などで、Inventoryに設定されているものとします。

In [2]:
hosts_file = './hosts'
!cat {hosts_file}

[cwhtest0001]
10.1.1.236 public_ip=35.76.33.195 ansible_ssh_user=centos ansible_ssh_private_key_file=~/.ssh/ansible_id_rsa
10.1.1.144 ansible_ssh_user=centos ansible_ssh_private_key_file=~/.ssh/ansible_id_rsa
10.1.1.157 ansible_ssh_user=centos ansible_ssh_private_key_file=~/.ssh/ansible_id_rsa


Ansible Inventory中に定義されているグループ( `[グループ名]` のように`[]`で囲まれた定義 )のうち、CoursewareHub構築に使うグループ名を指定します。

In [ ]:
target_group = 'cwhtest0001'

# 接続確認
%env ANSIBLE_INVENTORY={hosts_file}
!ansible -m ping {target_group}

# サーバーの役割の決定

以下の3つのサーバーに対して、役割を設定します。

In [4]:
hosts_stdout = !ansible -m ping {target_group}
hosts = [line.split()[0] for line in hosts_stdout if 'SUCCESS' in line]
hosts

['10.1.1.236', '10.1.1.144', '10.1.1.157']

設定例)

**NFSサーバーをCousewareHubのノードと分離する場合**

* `192.168.1.100` ... NFSサーバー
* `192.168.1.101` ... CoursewareHubノード1
* `192.168.1.102` ... CoursewareHubノード2

```
# NFSサーバーとして使うサーバー
nfs_server = '192.168.1.100'

# CoursewareHub masterとして使うサーバー ... public_ipがついているものである必要
master_server = '192.168.1.101'

# いずれにも指定されていないサーバーはnodeとして自動的に設定されます。
```

**NFSサーバーをCousewareHubのmasterノードと共有する場合**

* `192.168.1.101` ... CoursewareHubノード1 / NFSサーバー
* `192.168.1.102` ... CoursewareHubノード2


```
# NFSサーバーとして使うサーバー
nfs_server = '192.168.1.101'

# CoursewareHub masterとして使うサーバー ... public_ipがついているものである必要
master_server = '192.168.1.101'

# いずれにも指定されていないサーバーはnodeとして自動的に設定されます。
```


In [5]:
# NFSサーバーとして使うサーバー
nfs_server = '10.1.1.236'

# CoursewareHub masterとして使うサーバー ... public_ipがついているものである必要
master_server = '10.1.1.236'

# いずれにも指定されていないサーバーはnodeとして自動的に設定されます。

In [6]:
node_servers = list(set(hosts) - {nfs_server, master_server})
node_servers

['10.1.1.144', '10.1.1.157']

# 各種パラメータの決定

## NFSパラメータの決定

NFSサーバーは、複数のCoursewareHub環境から共有するように設定可能。個別のCoursewareHubを識別するための識別子を設定する。NFSのエクスポートするボリュームのパス名として利用される。

In [7]:
hubname = 'test01'

サーバーに関して、**NFSでの通信を別のIPアドレスで実施する場合**は以下のようにIP対応表を定義してください。

```
servicenet_ips = {
  'Public IP': 'Private IP'
}
```

In [8]:
servicenet_ips = {
}

IPアドレスがリスト中にあるかどうかを検証する。失敗する場合は↑の対応表を修正する。

In [9]:
for ip in servicenet_ips.keys():
    assert ip in hosts

## JupyterHubパラメータの決定

CoursewareHubのFQDNを設定する。

In [10]:
master_fqdn = 'cwhtest.example.nii.ac.jp'
master_fqdn

'nb12.ecloud.nii.ac.jp'

以下のパラメータ(5.2.1, 5.2.2, 5.2.3)は構築したいCoursewareHubの構成により選択する。

> はじめて構築する場合は、 5.2.1 フェデレーションに直接参加しないSPの場合 を選択するのが簡単です。

### フェデレーションに直接参加しないSPの場合

* idp-proxyを使用して、間接的に学認を使用する
* ローカルユーザーのみを使用する

In [11]:
auth_proxy_custom_image = 'auth-proxy-custom'
auth_proxy_custom_image

'auth-proxy-custom'

idp-proxyを利用する場合は、idp-proxyのFQDNを設定する。

idp-proxyを利用せず、ローカルユーザーのみを利用する場合は、空文字列を設定する。

In [12]:
#auth_fqdn = 'cwhidptest.example.nii.ac.jp'
auth_fqdn = ''
auth_fqdn

''

### フェデレーションに直接参加するSPの場合

idp-proxyを利用せずに、直接フェデレーションに参加する。

フェデレーションとクラウドゲートウェイの設定を行う。

In [13]:
#auth_proxy_custom_image = 'auth-proxy-fed'
#auth_proxy_custom_image

DSのサーバー名

In [14]:
#gakunin_ds_hostname = 'test-ds.gakunin.nii.ac.jp'
gakunin_ds_hostname = ''
gakunin_ds_hostname

''

SimpleSAMLphpのメターデータ更新(metarefresh)モジュールの設定ファイル名

https://github.com/NII-cloud-operation/CoursewareHub-LC_platform/tree/master/auth-proxy-fed/resources/simplesamlphp/config

In [15]:
#metarefresh_config = 'config-metarefresh-test.php'
metarefresh_config = ''
metarefresh_config

''

クラウドゲートウェイサーバーの設定

In [16]:
#cg_fqdn = 'sptest.cg.gakunin.jp'
cg_fqdn = ''
cg_fqdn

''

フェデレーションメタデータ検証用証明書の用意

In [17]:
#ds_metadata_signer_url = 'https://metadata.gakunin.nii.ac.jp/gakunin-test-signer-2011.cer'
ds_metadata_signer_url = ''
ds_metadata_signer_url

''

In [18]:
#ds_metadata_signer = 'gakunin-signer.cer'
ds_metadata_signer = ''
ds_metadata_signer

''

**テスト用クラウドゲートウェイを利用するときのみ**

クラウドゲートウェイメタデータの用意

学認から別途入手し、ローカルに配置済みとする。

In [19]:
## cgidp_metadata = 'sptestcgidp-metadata-self-signed-cert.xml.20200129'
## cgidp_metadata = 'sptestcgidp-metadata-self-signed-cert.xml'
#cgidp_metadata = 'sptestcgidp-metadata-self-signed-cert.xml.20200305'
cgidp_metadata = ''
cgidp_metadata

''

In [20]:
#!ls -l {cgidp_metadata}

In [21]:
#!ansible -b -m file -a 'path=/etc/jupyterhub/simplesamlphp/metadata/xml state=directory' {target_hub}
#!ansible -b -m copy -a 'src={cgidp_metadata} dest=/etc/jupyterhub/simplesamlphp/metadata/xml/cgidp-metadata.xml' {target_hub}

**本番系のクラウドゲートウェイを利用する場合**

メタデータディレクトリのみ作成する。

In [22]:
#!ansible -b -m file -a 'path=/etc/jupyterhub/simplesamlphp/metadata/xml state=directory' {target_hub}

### クラウドゲートウェイグループの設定

使用を許可するクラウドゲートウェイグループのグループIDを列挙する。

idp-proxyを使用するか、フェデレーションに直接参加しているときのみ有効。

ローカルユーザーのみの場合は、空の配列を指定する。

In [23]:
#cg_groups = [
#   'https://sptest.cg.gakunin.jp/gr/coursewarehub-test'
#]
cg_groups = []
cg_groups

[]

# Ansibleリソースの出力

このNotebookで設定された内容は、AnsibleのInventory, Ansibleのgroup_varsに保存することで、この後のNotebook実行で利用することができます。

In [24]:
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

'/tmp/tmpvxe80sv1'

In [25]:
nfs_server_with_ip = '{} servicenet_ip={}'.format(nfs_server, servicenet_ips[nfs_server] if nfs_server in servicenet_ips else nfs_server)
nfs_server_with_ip

'10.1.1.236 servicenet_ip=10.1.1.236'

In [26]:
master_server_with_ip = '{} servicenet_ip={}'.format(master_server, servicenet_ips[master_server] if master_server in servicenet_ips else master_server)
master_server_with_ip

'10.1.1.236 servicenet_ip=10.1.1.236'

In [27]:
node_servers_with_ips = ['{} servicenet_ip={}'.format(node_server, servicenet_ips[node_server] if node_server in servicenet_ips else node_server)
                        for node_server in node_servers]
node_servers_with_ips

['10.1.1.144 servicenet_ip=10.1.1.144', '10.1.1.157 servicenet_ip=10.1.1.157']

## Ansible Inventoryの出力

In [ ]:
node_servers_list_with_ips = '\n'.join(node_servers_with_ips)

print(f'''
[cwhub_nfs_server]
{nfs_server_with_ip}

[cwhub_master]
{master_server_with_ip}

[cwhub_nodes:children]
{target_group}_cwhub_nodes

[{target_group}_cwhub_nodes]
{node_servers_list_with_ips}
''')

AnsibleのInventoryに↑の定義を追加します。

Inventortyを変更する前に、現在の内容を作業ディレクトリにコピーしておく。

In [29]:
!cp -a {hosts_file} {work_dir}/inventory-old

In [30]:
# 編集が完了したら、Freezeして先に進める
assert False

AssertionError: 

編集の差分を確認しておく

In [ ]:
! ! diff -u {work_dir}/inventory-old {hosts_file}

## Ansible group_vars

In [33]:
nfspath = '/exported/{hubname}'.format(hubname=hubname)
nfspath

'/exported/test01'

In [34]:
import yaml

with open(os.path.join(work_dir, target_group), 'w') as f:
    f.write(yaml.dump({
        'nfspath': nfspath,
        'nfsserver': nfs_server,
        'exchange': '/exchange/',
        'nfsnodes_group': f'{target_group}_cwhub_nodes',
        'auth_proxy_custom_image': auth_proxy_custom_image,
        'master_fqdn': master_fqdn,
        'auth_fqdn': auth_fqdn,
        'gakunin_ds_hostname': gakunin_ds_hostname,
        'metarefresh_config': metarefresh_config,
        'cg_fqdn': cg_fqdn,
        'ds_metadata_signer_url': ds_metadata_signer_url,
        'ds_metadata_signer': ds_metadata_signer,
        'cgidp_metadata': cgidp_metadata,
        'cg_groups': cg_groups,
    }))

In [35]:
! ! diff -u {work_dir}/{target_group} {group_vars_dir}/{target_group}

--- /tmp/tmpvxe80sv1/cwhtest0001	2021-09-13 08:58:58.220321030 +0000
+++ ./group_vars/cwhtest0001	2021-09-10 09:16:58.955751009 +0000
@@ -11,4 +11,4 @@
 metarefresh_config: ''
 nfsnodes_group: cwhtest0001_cwhub_nodes
 nfspath: /exported/test01
-nfsserver: 10.1.1.236
+nfsserver: 10.1.1.79


In [36]:
!cp {work_dir}/{target_group} {group_vars_dir}/{target_group}

# 後始末

一時ファイルを削除する。

In [37]:
!rm -fr {work_dir}